# Endothelial receptor mining from GSE225948

This notebook inspects the public single-cell transcriptomic dataset associated with the StrokeVis resource and extracts gene-level endothelial receptor candidates for downstream peptide-shuttle design.

The aim is not to perform full formal differential-expression modelling. Instead, this notebook builds a practical omics-to-peptide-target workflow:

1. inspect the downloaded raw metadata and count files,
2. identify brain endothelial cell populations and experimental groups,
3. verify endothelial/BBB identity using canonical marker genes,
4. build EC pseudobulk expression profiles,
5. rank early stroke-responsive endothelial genes,
6. annotate candidate proteins with UniProt accessions and subcellular localisation,
7. classify putative surface accessibility,
8. export a receptor shortlist for downstream peptide generation and structure-aware triage.

Dataset/resource:

- **Title:** Brain and blood single-cell transcriptomic analysis in acute and subacute phases after experimental stroke  
- **Web resource:** https://anratherlab.shinyapps.io/strokevis/  
- **Public release:** June 14, 2023  
- **Local raw-data folder:** `../data/raw/GSE225948_RAW/`

Important scope note:

This analysis identifies **gene-level, stroke-responsive endothelial surface/membrane-associated candidates**, not validated receptor proteins. The exported candidates require follow-up validation of isoform-specific topology, extracellular-domain exposure, receptor complex state, internalisation capacity, mouse-human translation, and experimental binding.

In the wider prototype, this notebook provides the first stage of the computational decision chain:

**transcriptomic mining → receptor prioritisation → peptide-design constraints → candidate generation → receptor-aware structure triage → experimental handoff**

## 1. Inspect downloaded raw files

The raw StrokeVis/GSE225948 files were downloaded into the local project directory. This first step lists the available compressed CSV files and previews the first rows of each table to understand their structure before selecting the relevant metadata and expression matrices.

In [ ]:
# -------------------------
# 1.1 Imports and paths
# -------------------------

import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("../data/raw/GSE225948_RAW/")
files = sorted(data_dir.glob("*.csv.gz"))

print(f"Raw data directory: {data_dir}")
print(f"Number of .csv.gz files found: {len(files)}")

In [ ]:
# -------------------------
# 1.2 List available files
# -------------------------

for file in files:
    print("-", file.name)

In [ ]:
# -------------------------
# 1.3 Preview the first files
# -------------------------

for file in files[:5]:
    print("\n" + "=" * 80)
    print(file.name)

    df_preview = pd.read_csv(file, nrows=5)

    print("Preview shape:", df_preview.shape)
    display(df_preview)

## 2. Combine metadata across sequencing samples

Each downloaded sample includes a metadata table describing cell-level annotations such as tissue, treatment, sample description, parent cell type, and sub-cell type. These metadata files are concatenated into one table so that endothelial cells and stroke-related experimental groups can be selected consistently across samples.

In [ ]:
# -------------------------
# 2.1 Load and concatenate metadata files
# -------------------------

metadata_files = sorted(data_dir.glob("*metadata.csv.gz"))

print(f"Metadata files found: {len(metadata_files)}")
for f in metadata_files:
    print("-", f.name)

all_meta = []

for f in metadata_files:
    meta = pd.read_csv(f)
    meta["source_file"] = f.name
    all_meta.append(meta)

meta_all = pd.concat(all_meta, ignore_index=True)

print("Combined metadata shape:", meta_all.shape)
display(meta_all.head())

In [ ]:
# -------------------------
# 2.2 Inspect metadata columns
# -------------------------

print("Metadata columns:")
for col in meta_all.columns:
    print("-", col)

In [ ]:
# -------------------------
# 2.3 Inspect tissue and treatment labels
# -------------------------

print("Tissues:")
display(meta_all["tissue"].value_counts().to_frame("n_cells"))

print("Treatments:")
display(meta_all["treatment"].value_counts().to_frame("n_cells"))

print("Sample descriptions:")
display(meta_all["Sample_description"].value_counts().to_frame("n_cells"))

In [ ]:
# -------------------------
# 2.4 Inspect cell-type annotations
# -------------------------

print("Parent cell types:")
display(meta_all["parent"].value_counts().to_frame("n_cells"))

print("Top sub-cell types:")
display(meta_all["sub.celltype"].value_counts().head(50).to_frame("n_cells"))

## 3. Select brain endothelial cells

The receptor-mining workflow focuses on endothelial cells from brain tissue because these cells represent the blood-brain barrier and vascular interface relevant to peptide-shuttle design. This section filters the combined metadata to brain endothelial cells and checks whether enough cells are available across treatments, subclusters, and source files.

In [ ]:
# -------------------------
# 3.1 Filter brain endothelial cells
# -------------------------

ec_meta = meta_all[
    (meta_all["tissue"] == "brain")
    & (meta_all["parent"] == "EC")
].copy()

print("Brain endothelial metadata shape:", ec_meta.shape)
display(ec_meta.head())

In [ ]:
# -------------------------
# 3.2 Inspect endothelial cells by treatment
# -------------------------

display(
    ec_meta["treatment"]
    .value_counts()
    .to_frame("n_cells")
)

In [ ]:
# -------------------------
# 3.3 Inspect endothelial subclusters
# -------------------------

display(
    ec_meta["sub.celltype"]
    .value_counts()
    .to_frame("n_cells")
)

In [ ]:
# -------------------------
# 3.4 Cross-tabulate endothelial subclusters by treatment
# -------------------------

ec_subcluster_treatment_counts = pd.crosstab(
    ec_meta["sub.celltype"],
    ec_meta["treatment"],
)

display(ec_subcluster_treatment_counts)

In [ ]:
# -------------------------
# 3.5 Inspect endothelial cells by treatment and source file
# -------------------------

ec_by_source = (
    ec_meta
    .groupby(["treatment", "source_file"])
    .size()
    .reset_index(name="n_cells")
    .sort_values(["treatment", "source_file"])
)

display(ec_by_source)

## 4. Validate endothelial identity using marker genes

Before prioritising receptor candidates, the endothelial-cell annotation is checked using known endothelial and blood-brain-barrier markers. The expectation is that brain EC-labelled cells should express endothelial markers such as `Cldn5`, `Pecam1`, `Cdh5`, `Kdr`, `Flt1`, `Tek`, `Slc2a1`, and `Mfsd2a`, while non-endothelial markers such as immune, pericyte, mural, and astrocyte markers should remain comparatively low.

This step is a sanity check to support that the EC subset used for downstream receptor mining is biologically reasonable.

In [ ]:
# -------------------------
# 4.1 Define endothelial and negative-control marker genes
# -------------------------

endothelial_markers = [
    "Cldn5", "Pecam1", "Cdh5", "Kdr", "Flt1", "Tek",
    "Slc2a1", "Mfsd2a", "Abcb1a", "Abcg2", "Tfrc", "Slc7a5",
    "Vwf", "Cav1", "Plvap",
]

negative_markers = [
    "Ptprc",   # immune cells
    "Aif1",   # microglia/macrophages
    "Pdgfrb", # pericytes
    "Rgs5",   # mural/pericyte
    "Gfap",   # astrocytes
    "Aqp4",   # astrocyte/endfoot-associated
]

marker_genes = endothelial_markers + negative_markers

print("Endothelial markers:", len(endothelial_markers))
print("Negative-control markers:", len(negative_markers))
print("Total marker genes:", len(marker_genes))

In [ ]:
# -------------------------
# 4.2 Helper function to calculate EC marker expression per sample
# -------------------------

def load_gene_panel_for_ec(meta_path, genes):
    counts_path = Path(str(meta_path).replace("_metadata.csv.gz", "_counts.csv.gz"))

    if not counts_path.exists():
        print(f"Counts file missing for: {meta_path.name}")
        return None

    meta = pd.read_csv(meta_path)
    meta["Unnamed: 0"] = meta["Unnamed: 0"].astype(str)

    ec_meta_sample = meta[
        (meta["tissue"] == "brain")
        & (meta["parent"] == "EC")
    ].copy()

    if ec_meta_sample.empty:
        return None

    ec_cells = ec_meta_sample["Unnamed: 0"].tolist()

    counts = pd.read_csv(counts_path, index_col=0)
    counts.index = counts.index.astype(str)
    counts.columns = counts.columns.astype(str)

    available_genes = [g for g in genes if g in counts.index]
    available_cells = [c for c in ec_cells if c in counts.columns]

    print(
        meta_path.name,
        "| EC cells:", len(ec_cells),
        "| matched cells:", len(available_cells),
        "| matched genes:", len(available_genes),
    )

    if len(available_genes) == 0 or len(available_cells) == 0:
        return None

    expr = counts.loc[available_genes, available_cells]

    out = expr.mean(axis=1).reset_index()
    out.columns = ["gene", "mean_expression"]

    out["treatment"] = ec_meta_sample["treatment"].iloc[0]
    out["source_file"] = meta_path.name
    out["n_ec_cells"] = len(available_cells)

    return out

In [ ]:
# -------------------------
# 4.3 Calculate marker expression across EC samples
# -------------------------

all_marker_expr = []

for meta_path in sorted(data_dir.glob("*metadata.csv.gz")):
    result = load_gene_panel_for_ec(meta_path, marker_genes)
    if result is not None:
        all_marker_expr.append(result)

if len(all_marker_expr) == 0:
    raise ValueError("No marker expression results found. Check data_dir and filenames.")

marker_expr = pd.concat(all_marker_expr, ignore_index=True)

print("Marker expression table shape:", marker_expr.shape)
display(marker_expr.head())

In [ ]:
# -------------------------
# 4.4 Summarise marker expression by treatment
# -------------------------

marker_summary = (
    marker_expr
    .groupby(["gene", "treatment"])["mean_expression"]
    .mean()
    .reset_index()
    .pivot(index="gene", columns="treatment", values="mean_expression")
)

display(marker_summary)

In [ ]:
# -------------------------
# 4.5 Save EC marker validation outputs
# -------------------------

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

marker_summary_path = processed_dir / "ec_marker_summary.csv"
marker_expr_path = processed_dir / "ec_marker_expression_by_sample.csv"

marker_summary.to_csv(marker_summary_path)
marker_expr.to_csv(marker_expr_path, index=False)

print("Saved marker summary:", marker_summary_path)
print("Saved marker expression by sample:", marker_expr_path)

## 5. Build EC pseudobulk expression and compare stroke time points

To prioritise stroke-induced endothelial receptor candidates, a sample-level EC pseudobulk matrix is created. For each sample, the expression of each gene is averaged across brain endothelial cells. This converts the cell-level count matrices into one expression profile per biological sample.

The pseudobulk matrix is then used to compare early and subacute stroke time points against sham controls:

- `D02` versus `Sham`
- `D14` versus `Sham`

The resulting tables provide simple effect-size, log2 fold-change proxy, and Welch t-test statistics for each gene. These outputs are used later for receptor prioritisation rather than formal differential-expression claims.

In [ ]:
# -------------------------
# 5.1 Build EC pseudobulk expression matrix
# -------------------------

from scipy.stats import ttest_ind

out_dir = Path("../data/processed/")
out_dir.mkdir(parents=True, exist_ok=True)

sample_expr = []
sample_info = []

for meta_path in sorted(data_dir.glob("*metadata.csv.gz")):
    counts_path = Path(str(meta_path).replace("_metadata.csv.gz", "_counts.csv.gz"))

    if not counts_path.exists():
        continue

    meta = pd.read_csv(meta_path)
    meta["Unnamed: 0"] = meta["Unnamed: 0"].astype(str)

    # Keep only brain endothelial cells
    ec_meta_sample = meta[
        (meta["tissue"] == "brain")
        & (meta["parent"] == "EC")
    ].copy()

    if ec_meta_sample.empty:
        continue

    ec_cells = ec_meta_sample["Unnamed: 0"].tolist()

    counts = pd.read_csv(counts_path, index_col=0)
    counts.index = counts.index.astype(str)
    counts.columns = counts.columns.astype(str)

    available_cells = [c for c in ec_cells if c in counts.columns]

    if len(available_cells) == 0:
        continue

    # Pseudobulk: average expression of each gene across EC cells in this sample
    mean_expr = counts[available_cells].mean(axis=1)

    sample_id = meta_path.name.replace("_metadata.csv.gz", "")
    mean_expr.name = sample_id

    sample_expr.append(mean_expr)

    sample_info.append({
        "sample_id": sample_id,
        "source_file": meta_path.name,
        "treatment": ec_meta_sample["treatment"].iloc[0],
        "age": ec_meta_sample["age"].iloc[0],
        "sex": ec_meta_sample["sex"].iloc[0],
        "n_ec_cells": len(available_cells),
    })

ec_pseudobulk = pd.concat(sample_expr, axis=1)
ec_sample_info = pd.DataFrame(sample_info)

print("Pseudobulk matrix shape:", ec_pseudobulk.shape)
display(ec_sample_info)

In [ ]:
# -------------------------
# 5.2 Save EC pseudobulk outputs
# -------------------------

ec_pseudobulk_path = out_dir / "ec_pseudobulk_mean_expression.csv"
ec_sample_info_path = out_dir / "ec_sample_info.csv"

ec_pseudobulk.to_csv(ec_pseudobulk_path)
ec_sample_info.to_csv(ec_sample_info_path, index=False)

print("Saved EC pseudobulk matrix:", ec_pseudobulk_path)
print("Saved EC sample information:", ec_sample_info_path)

In [ ]:
# -------------------------
# 5.3 Define treatment comparison helper
# -------------------------

def compare_treatments(expr, info, case, control="Sham"):
    case_samples = info.loc[info["treatment"] == case, "sample_id"].tolist()
    control_samples = info.loc[info["treatment"] == control, "sample_id"].tolist()

    print(f"{case} samples:", len(case_samples))
    print(f"{control} samples:", len(control_samples))

    if len(case_samples) == 0 or len(control_samples) == 0:
        raise ValueError(f"Missing samples for comparison: {case} vs {control}")

    x_case = expr[case_samples]
    x_control = expr[control_samples]

    mean_case = x_case.mean(axis=1)
    mean_control = x_control.mean(axis=1)

    effect = mean_case - mean_control

    eps = 1e-6
    log2fc_proxy = np.log2((mean_case + eps) / (mean_control + eps))

    pvals = ttest_ind(
        x_case.T,
        x_control.T,
        axis=0,
        equal_var=False,
        nan_policy="omit",
    ).pvalue

    out = pd.DataFrame({
        "gene": expr.index,
        f"mean_{case}": mean_case.values,
        f"mean_{control}": mean_control.values,
        f"effect_{case}_vs_{control}": effect.values,
        f"log2fc_proxy_{case}_vs_{control}": log2fc_proxy.values,
        f"pvalue_{case}_vs_{control}": pvals,
    })

    return out.sort_values(
        f"effect_{case}_vs_{control}",
        ascending=False,
    ).reset_index(drop=True)

In [ ]:
# -------------------------
# 5.4 Compare D02 and D14 versus Sham
# -------------------------

de_d02 = compare_treatments(
    ec_pseudobulk,
    ec_sample_info,
    case="D02",
)

de_d14 = compare_treatments(
    ec_pseudobulk,
    ec_sample_info,
    case="D14",
)

de_d02_path = out_dir / "ec_DE_D02_vs_Sham.csv"
de_d14_path = out_dir / "ec_DE_D14_vs_Sham.csv"

de_d02.to_csv(de_d02_path, index=False)
de_d14.to_csv(de_d14_path, index=False)

print("Saved D02 comparison:", de_d02_path)
print("Saved D14 comparison:", de_d14_path)

print("Top D02-induced EC genes:")
display(de_d02.head(20))

print("Top D14-induced EC genes:")
display(de_d14.head(20))

## 6. Inspect known BBB receptor, transporter, and endothelial surface candidates

Before mining the full transcriptome for novel receptor candidates, a curated target panel is inspected. This panel includes known or plausible BBB transport receptors, endothelial receptors, barrier identity markers, efflux transporters, inflammatory adhesion molecules, and stroke-induced surface candidates.

This step serves as a biological sanity check. It asks whether known BBB/endothelial genes are detected in the EC pseudobulk table and whether they are preserved, induced, or suppressed after stroke. The resulting table is used for interpretation and context, while the final receptor shortlist is generated in later sections using broader surface-accessibility and induction criteria.

In [ ]:
# -------------------------
# 6.1 Define curated BBB/receptor target panel
# -------------------------

target_panel = {
    # known / plausible BBB shuttle or transport-relevant candidates
    "Tfrc": "known BBB receptor-mediated uptake candidate",
    "Lrp1": "endocytic BBB transport receptor",
    "Insr": "receptor-mediated transcytosis precedent",
    "Igf1r": "receptor-mediated transcytosis precedent",
    "Bsg": "membrane glycoprotein, vascular/inflammatory relevance",
    "Cd36": "scavenger receptor, vascular/endothelial relevance",
    "Scarb1": "scavenger receptor, lipid transport relevance",

    # BBB transporters / barrier identity
    "Slc2a1": "BBB glucose transporter and EC identity marker",
    "Slc7a5": "BBB amino-acid transporter",
    "Mfsd2a": "BBB lipid transporter and barrier identity marker",
    "Abcb1a": "BBB efflux transporter",
    "Abcg2": "BBB efflux transporter",
    "Slco1a4": "brain endothelial solute carrier candidate",

    # vascular / endothelial receptors
    "Kdr": "VEGF receptor, endothelial receptor",
    "Flt1": "VEGF receptor, endothelial receptor",
    "Tek": "Tie2 endothelial receptor",
    "Eng": "endothelial accessory receptor",

    # stroke/inflammatory endothelial activation
    "Icam1": "activated endothelial adhesion receptor",
    "Vcam1": "activated endothelial adhesion receptor",
    "Sele": "activated endothelial adhesion receptor",
    "Plvap": "vascular permeability-associated membrane protein",
    "Ly6a": "stroke-induced endothelial surface marker candidate",
    "Ly6c1": "stroke-associated vascular/immune interface marker",

    # membrane/cell-surface candidates from top results
    "Igfbp7": "secreted/perivascular EC-associated factor, target-context marker",
    "Itm2a": "membrane protein candidate",
}

print("Number of curated targets:", len(target_panel))

In [ ]:
# -------------------------
# 6.2 Define curated target-table helper
# -------------------------

def make_target_table(de_d02, de_d14, target_panel):
    d02_cols = [
        "gene",
        "mean_D02",
        "mean_Sham",
        "effect_D02_vs_Sham",
        "log2fc_proxy_D02_vs_Sham",
        "pvalue_D02_vs_Sham",
    ]

    d14_cols = [
        "gene",
        "mean_D14",
        "mean_Sham",
        "effect_D14_vs_Sham",
        "log2fc_proxy_D14_vs_Sham",
        "pvalue_D14_vs_Sham",
    ]

    d02 = de_d02[d02_cols].copy()
    d14 = de_d14[d14_cols].copy()

    merged = d02.merge(
        d14,
        on="gene",
        how="outer",
        suffixes=("_D02table", "_D14table"),
    )

    panel = pd.DataFrame({
        "gene": list(target_panel.keys()),
        "BBB_relevance": list(target_panel.values()),
    })

    out = panel.merge(merged, on="gene", how="left")

    def classify_response(row):
        d02_effect = row.get("effect_D02_vs_Sham", 0)
        d14_effect = row.get("effect_D14_vs_Sham", 0)

        if pd.isna(d02_effect):
            d02_effect = 0
        if pd.isna(d14_effect):
            d14_effect = 0

        if d02_effect > 0.5 and d14_effect > 0.5:
            return "persistent stroke-induced"
        if d02_effect > 0.5 and d14_effect <= 0.5:
            return "early D02-induced"
        if d02_effect <= 0.5 and d14_effect > 0.5:
            return "late D14-induced"
        if d02_effect < -0.5 or d14_effect < -0.5:
            return "stroke-suppressed"
        return "preserved / weakly changed"

    out["stroke_response"] = out.apply(classify_response, axis=1)

    out["cell_source_evidence"] = (
        "brain EC pseudobulk; EC identity verified by canonical endothelial and BBB markers"
    )

    target_constraints = {
        "Tfrc": "bind extracellular receptor region; avoid disrupting iron transport",
        "Lrp1": "exploit endocytic uptake; avoid broad off-target uptake",
        "Insr": "avoid metabolic agonism while preserving transport",
        "Igf1r": "avoid growth-factor-like signalling activation",
        "Bsg": "target extracellular domain; check inflammatory/off-target expression",
        "Cd36": "avoid broad scavenger-receptor off-target effects",
        "Scarb1": "balance lipid-transport biology and endothelial specificity",
        "Slc2a1": "use mainly as BBB identity marker, not primary peptide target",
        "Slc7a5": "transporter biology relevant; peptide binding may be difficult",
        "Mfsd2a": "BBB identity marker; targetability requires caution",
        "Abcb1a": "efflux transporter; not ideal as shuttle target",
        "Abcg2": "efflux transporter; not ideal as shuttle target",
        "Slco1a4": "evaluate extracellular accessibility and species translation",
        "Kdr": "avoid angiogenic signalling activation",
        "Flt1": "avoid VEGF-pathway perturbation",
        "Tek": "avoid Tie2 pathway activation unless intended",
        "Eng": "evaluate vascular specificity and signalling risk",
        "Icam1": "stroke-activated targeting; avoid systemic immune adhesion effects",
        "Vcam1": "stroke-activated targeting; avoid broad inflammatory endothelium",
        "Sele": "stroke-activated targeting; likely inflammation-state specific",
        "Plvap": "vascular permeability marker; assess BBB specificity",
        "Ly6a": "strong mouse stroke EC signal; check human translational relevance",
        "Ly6c1": "stroke-associated but immune/vascular specificity must be checked",
        "Igfbp7": "use as EC/stroke context marker more than direct shuttle receptor",
        "Itm2a": "evaluate membrane topology and extracellular accessibility",
    }

    out["target_constraint"] = out["gene"].map(target_constraints)

    response_order = {
        "early D02-induced": 0,
        "persistent stroke-induced": 1,
        "late D14-induced": 2,
        "preserved / weakly changed": 3,
        "stroke-suppressed": 4,
    }

    out["response_order"] = out["stroke_response"].map(response_order)

    out = out.sort_values(
        ["response_order", "effect_D02_vs_Sham"],
        ascending=[True, False],
    ).drop(columns="response_order")

    return out.reset_index(drop=True)

In [ ]:
# -------------------------
# 6.3 Build curated target table
# -------------------------

target_table = make_target_table(
    de_d02=de_d02,
    de_d14=de_d14,
    target_panel=target_panel,
)

display(target_table)

In [ ]:
# -------------------------
# 6.4 Save curated target table
# -------------------------

target_table_path = out_dir / "ec_receptor_transporter_target_table.csv"

target_table.to_csv(target_table_path, index=False)

print("Saved curated receptor/transporter target table:", target_table_path)

## 7. Identify data-driven stroke-responsive EC candidates

After inspecting a curated BBB/receptor panel, a broader data-driven screen is performed across the EC pseudobulk comparison results. The aim is to identify genes that are strongly induced after stroke and expressed in endothelial cells, before applying surface-accessibility and UniProt annotation filters in later sections.

This step removes obvious housekeeping or non-target-like genes, such as ribosomal, mitochondrial, haemoglobin, predicted, and microRNA genes. The remaining genes are filtered by minimum case expression and stroke-induced effect size.

In [ ]:
# -------------------------
# 7.1 Define data-driven candidate filter
# -------------------------

import re

def filter_data_driven_candidates(
    de,
    case,
    min_mean_case=0.05,
    min_effect=0.25,
):
    effect_col = f"effect_{case}_vs_Sham"
    mean_col = f"mean_{case}"
    logfc_col = f"log2fc_proxy_{case}_vs_Sham"

    df = de.copy()

    # Remove obvious non-target / housekeeping-style genes
    bad_patterns = [
        r"^Rpl",   # ribosomal large subunit
        r"^Rps",   # ribosomal small subunit
        r"^mt-",   # mitochondrial
        r"^Hba",   # haemoglobin alpha
        r"^Hbb",   # haemoglobin beta
        r"^Gm\d+", # predicted genes
        r"^Mir",   # microRNAs
    ]

    bad_exact = {
        "Malat1",
        "Actb",
        "Gapdh",
        "Tmsb4x",
    }

    pattern = "|".join(bad_patterns)

    df = df[
        ~df["gene"].str.contains(pattern, regex=True, na=False)
        & ~df["gene"].isin(bad_exact)
    ].copy()

    # Keep genes with meaningful expression and stroke induction
    df = df[
        (df[mean_col] >= min_mean_case)
        & (df[effect_col] >= min_effect)
    ].copy()

    # Sort by absolute effect first, then fold-change proxy
    df = df.sort_values(
        [effect_col, logfc_col],
        ascending=False,
    ).reset_index(drop=True)

    return df

In [ ]:
# -------------------------
# 7.2 Apply data-driven filtering to D02 and D14
# -------------------------

d02_candidates = filter_data_driven_candidates(
    de=de_d02,
    case="D02",
)

d14_candidates = filter_data_driven_candidates(
    de=de_d14,
    case="D14",
)

print("D02 candidates:", d02_candidates.shape)
print("D14 candidates:", d14_candidates.shape)

print("Top data-driven D02 EC candidates:")
display(d02_candidates.head(50))

print("Top data-driven D14 EC candidates:")
display(d14_candidates.head(50))

In [ ]:
# -------------------------
# 7.3 Save data-driven candidate tables
# -------------------------

d02_candidates_path = out_dir / "data_driven_D02_EC_candidates.csv"
d14_candidates_path = out_dir / "data_driven_D14_EC_candidates.csv"

d02_candidates.head(200).to_csv(d02_candidates_path, index=False)
d14_candidates.head(200).to_csv(d14_candidates_path, index=False)

print("Saved D02 data-driven candidates:", d02_candidates_path)
print("Saved D14 data-driven candidates:", d14_candidates_path)

## 8. Annotate data-driven candidates with UniProt surface-accessibility information

The data-driven D02 and D14 candidate lists include many genes that are stroke-responsive in endothelial cells but are not necessarily suitable peptide-shuttle targets. For peptide targeting, the protein should ideally be cell-surface accessible, membrane-associated with an extracellular domain, or otherwise biologically relevant to the extracellular endothelial environment.

This section queries UniProt for the top data-driven candidates and extracts canonical protein annotations, subcellular localisation, and keywords. These annotations are then converted into broad accessibility classes:

- `surface_accessible`: strongest class for direct peptide targeting
- `membrane_unknown_location`: possible target, requires topology/domain inspection
- `secreted_or_extracellular`: useful biology or context marker, but usually not a direct receptor
- `intracellular_or_organelle`: rejected for extracellular peptide targeting
- `unknown`: requires manual inspection

In [ ]:
# -------------------------
# 8.1 Define UniProt query and annotation helpers
# -------------------------

import time
import requests

def safe_text(x):
    if x is None or pd.isna(x):
        return ""
    return str(x)


def query_uniprot_mouse_gene(gene):
    url = "https://rest.uniprot.org/uniprotkb/search"
    fields = "accession,gene_names,protein_name,cc_subcellular_location,keyword"

    queries = [
        f"gene_exact:{gene} AND organism_id:10090 AND reviewed:true",
        f"gene_exact:{gene} AND organism_id:10090",
    ]

    for query in queries:
        r = requests.get(
            url,
            params={
                "query": query,
                "format": "json",
                "size": 1,
                "fields": fields,
            },
            timeout=30,
        )
        r.raise_for_status()
        results = r.json().get("results", [])

        if results:
            entry = results[0]
            break
    else:
        return {
            "gene": gene,
            "uniprot_accession": None,
            "protein_name": None,
            "subcellular_location_text": None,
            "keywords_text": None,
            "uniprot_status": "not_found",
        }

    protein_desc = entry.get("proteinDescription", {})
    protein_name = (
        protein_desc
        .get("recommendedName", {})
        .get("fullName", {})
        .get("value")
    )

    if protein_name is None and protein_desc.get("submissionNames"):
        protein_name = (
            protein_desc["submissionNames"][0]
            .get("fullName", {})
            .get("value")
        )

    locations = []

    for comment in entry.get("comments", []):
        if comment.get("commentType") == "SUBCELLULAR LOCATION":
            for loc in comment.get("subcellularLocations", []):
                pieces = [
                    loc.get("location", {}).get("value"),
                    loc.get("topology", {}).get("value"),
                    loc.get("orientation", {}).get("value"),
                ]
                pieces = [p for p in pieces if p]

                if pieces:
                    locations.append("; ".join(pieces))

    keywords = [
        kw["name"]
        for kw in entry.get("keywords", [])
        if kw.get("name")
    ]

    return {
        "gene": gene,
        "uniprot_accession": entry.get("primaryAccession"),
        "protein_name": protein_name,
        "subcellular_location_text": " | ".join(locations) if locations else None,
        "keywords_text": " | ".join(keywords) if keywords else None,
        "uniprot_status": "found",
    }

In [ ]:
# -------------------------
# 8.2 Define surface-accessibility classifier
# -------------------------

def classify_surface_accessibility(location_text, keywords_text):
    text = f"{safe_text(location_text)} {safe_text(keywords_text)}".lower()

    surface_terms = [
        "cell membrane",
        "plasma membrane",
        "cell surface",
        "external side of plasma membrane",
        "extracellular side",
        "gpi-anchor",
        "glycosylphosphatidylinositol",
        "lipid-anchor",
    ]

    extracellular_terms = [
        "secreted",
        "extracellular space",
        "extracellular matrix",
        "extracellular region",
    ]

    intracellular_terms = [
        "mitochondrion",
        "mitochondrial",
        "endoplasmic reticulum",
        "sarcoplasmic reticulum",
        "golgi",
        "lysosome",
        "endosome",
        "cytoplasmic vesicle",
        "nucleus",
        "nuclear",
        "cytoplasm",
        "cytosol",
        "ribosome",
        "peroxisome",
    ]

    has_surface = any(t in text for t in surface_terms)
    has_extra = any(t in text for t in extracellular_terms)
    has_intra = any(t in text for t in intracellular_terms)
    has_membrane = "membrane" in text or "transmembrane" in text

    if has_surface:
        return "surface_accessible"
    if has_extra and has_intra:
        return "secreted_or_extracellular_mixed"
    if has_extra:
        return "secreted_or_extracellular"
    if has_intra:
        return "intracellular_or_organelle"
    if has_membrane:
        return "membrane_unknown_location"

    return "unknown"


def targetability_comment(surface_class):
    comments = {
        "surface_accessible": "best class for direct peptide-shuttle targeting",
        "membrane_unknown_location": "possible target, but needs topology/extracellular-domain check",
        "secreted_or_extracellular": "useful EC/stroke context marker, usually not a direct receptor",
        "secreted_or_extracellular_mixed": "possible extracellular context marker, but not a clean direct receptor",
        "unknown": "manual annotation required",
        "intracellular_or_organelle": "reject for extracellular peptide targeting",
    }

    return comments.get(surface_class, "manual annotation required")


def targetability_score(surface_class):
    scores = {
        "surface_accessible": 4,
        "membrane_unknown_location": 2,
        "secreted_or_extracellular": 1,
        "secreted_or_extracellular_mixed": 0.5,
        "unknown": 0,
        "intracellular_or_organelle": -1,
    }

    return scores.get(surface_class, 0)

In [ ]:
# -------------------------
# 8.3 Annotate candidates with UniProt
# -------------------------

def annotate_with_uniprot(df, gene_col="gene", sleep=0.2):
    rows = []

    for gene in df[gene_col].dropna().astype(str).unique():
        try:
            row = query_uniprot_mouse_gene(gene)
        except Exception as e:
            row = {
                "gene": gene,
                "uniprot_accession": None,
                "protein_name": None,
                "subcellular_location_text": None,
                "keywords_text": None,
                "uniprot_status": "error",
                "uniprot_error": str(e),
            }

        rows.append(row)
        time.sleep(sleep)

    ann = pd.DataFrame(rows)

    ann["surface_accessibility"] = ann.apply(
        lambda r: classify_surface_accessibility(
            r.get("subcellular_location_text"),
            r.get("keywords_text"),
        ),
        axis=1,
    )

    ann["targetability_score"] = ann["surface_accessibility"].apply(
        targetability_score
    )

    ann["targetability_comment"] = ann["surface_accessibility"].apply(
        targetability_comment
    )

    return df.merge(ann, on="gene", how="left")


d02_annotated = annotate_with_uniprot(d02_candidates.head(100))
d14_annotated = annotate_with_uniprot(d14_candidates.head(100))

d02_annotated_path = out_dir / "D02_candidates_uniprot_annotated.csv"
d14_annotated_path = out_dir / "D14_candidates_uniprot_annotated.csv"

d02_annotated.to_csv(d02_annotated_path, index=False)
d14_annotated.to_csv(d14_annotated_path, index=False)

print("Saved D02 UniProt annotations:", d02_annotated_path)
print("Saved D14 UniProt annotations:", d14_annotated_path)

In [ ]:
# -------------------------
# 8.4 Inspect surface-accessibility classes
# -------------------------

print("D02 surface-accessibility classes:")
display(
    d02_annotated["surface_accessibility"]
    .value_counts(dropna=False)
    .to_frame("n_genes")
)

print("D14 surface-accessibility classes:")
display(
    d14_annotated["surface_accessibility"]
    .value_counts(dropna=False)
    .to_frame("n_genes")
)

## 9. Prioritise peptide-shuttle receptor candidates

The D02 response is used as the primary prioritisation window because the aim is to identify early stroke-induced endothelial surface candidates. Candidate genes are first filtered to keep proteins with plausible extracellular or membrane relevance based on UniProt annotation. Manual biological prioritisation is then applied to separate likely peptide-target candidates from secondary candidates, extracellular context markers, and false-positive surface annotations.

The final ranking combines:

- EC pseudobulk expression after D02 stroke,
- D02 induction effect size,
- log2 fold-change proxy,
- UniProt surface-accessibility class,
- and manual biological interpretation.

In [ ]:
# -------------------------
# 9.1 Build D02 candidate pool from UniProt-annotated genes
# -------------------------

candidate_classes = [
    "surface_accessible",
    "membrane_unknown_location",
    "secreted_or_extracellular",
    "secreted_or_extracellular_mixed",
    "unknown",
]

candidate_pool = d02_annotated[
    d02_annotated["surface_accessibility"].isin(candidate_classes)
].copy()

print("Candidate pool shape:", candidate_pool.shape)

display(
    candidate_pool[
        [
            "gene",
            "protein_name",
            "mean_D02",
            "effect_D02_vs_Sham",
            "log2fc_proxy_D02_vs_Sham",
            "surface_accessibility",
            "targetability_comment",
        ]
    ].head(20)
)

In [ ]:
# -------------------------
# 9.2 Classify D02 stroke response
# -------------------------

def classify_stroke_response(row):
    effect = row["effect_D02_vs_Sham"]
    logfc = row["log2fc_proxy_D02_vs_Sham"]
    mean_d02 = row["mean_D02"]

    if effect >= 1.0 and logfc >= 1.0:
        return "strong early D02-induced"
    if effect >= 0.5 and logfc >= 0.5:
        return "moderate early D02-induced"
    if mean_d02 >= 1.0 and abs(effect) < 0.5:
        return "highly expressed / preserved"
    if effect > 0:
        return "weakly D02-induced"

    return "not D02-induced"


candidate_pool["stroke_response"] = candidate_pool.apply(
    classify_stroke_response,
    axis=1,
)

display(
    candidate_pool["stroke_response"]
    .value_counts()
    .to_frame("n_genes")
)

In [ ]:
# -------------------------
# 9.3 Manual biological prioritisation
# -------------------------

primary = {
    "Ly6a", "Ly6e", "Tm4sf1", "Esam",
    "Cd200", "Itgb1", "Fxyd5", "Ecscr",
}

secondary = {
    "Ly6c1", "Ifitm3", "Ifitm2", "Itm2a",
    "Tmem252", "Ramp2", "Cd81", "Pecam1",
}

context = {
    "Igfbp7", "Sparc", "Egfl7", "Edn1", "Col4a1", "Col4a2",
}

reject_surface = {
    "Hsp90ab1", "Vim", "Eef1a1", "Glul", "Hspa5", "Calr",
    "Gnas", "Hspa8", "Rack1", "Msn", "Cdc42", "Cfl1", "Rab11a",
}


def manual_priority(gene):
    if gene in primary:
        return "primary peptide-target candidate"
    if gene in secondary:
        return "secondary candidate / needs validation"
    if gene in context:
        return "stroke EC context marker, not direct receptor"
    if gene in reject_surface:
        return "reject despite surface annotation"

    return "not prioritised"


manual_notes = {
    "Ly6a": "Strongest D02-induced GPI-anchored surface signal; attractive mouse stroke-BBB target but human translation must be checked.",
    "Ly6e": "GPI-anchored surface protein with early induction; possible Ly6-family target.",
    "Tm4sf1": "Transmembrane protein with strong D02 induction; extracellular loops/topology need validation.",
    "Esam": "Endothelial-selective adhesion molecule; surface accessible and EC-relevant.",
    "Cd200": "Single-pass membrane glycoprotein with significant early induction; extracellular domain likely accessible.",
    "Itgb1": "Surface integrin receptor; targetable but broad expression/off-target risk.",
    "Fxyd5": "Single-pass membrane protein with early induction; needs BBB/endothelial specificity check.",
    "Ecscr": "Endothelial cell-specific surface regulator with strong logFC and significant D02 response.",
    "Ly6c1": "Strong surface signal but immune/vascular specificity and translational relevance need caution.",
    "Ifitm3": "Induced membrane protein but more interferon/stress-associated than classic shuttle receptor.",
    "Ifitm2": "Induced membrane protein; similar caution as Ifitm3.",
    "Itm2a": "Membrane protein candidate; extracellular topology and BBB specificity need validation.",
    "Tmem252": "Strong D02 induction; membrane topology and function unclear.",
    "Ramp2": "Surface receptor-modifying protein; relevant but signalling biology may complicate targeting.",
    "Cd81": "Surface tetraspanin; targetable but broad expression risk.",
    "Pecam1": "Endothelial surface marker; useful control/reference but broad vascular expression.",
    "Igfbp7": "Strong EC/stroke marker but secreted, therefore not a clean direct receptor.",
    "Sparc": "Secreted ECM/remodelling marker; useful context marker.",
    "Egfl7": "Secreted endothelial/angiogenic context marker, not direct receptor.",
    "Edn1": "Secreted vasoactive peptide; context marker, not shuttle receptor.",
    "Col4a1": "Extracellular matrix marker, not direct shuttle receptor.",
    "Col4a2": "Extracellular matrix marker, not direct shuttle receptor.",
}


def final_rationale(priority):
    if priority == "primary peptide-target candidate":
        return (
            "prioritised for peptide design; inspect extracellular domain/topology, "
            "internalisation, BBB specificity, and human orthology"
        )

    if priority == "secondary candidate / needs validation":
        return (
            "possible peptide-design target; requires validation of specificity, "
            "topology, or translational relevance"
        )

    if priority == "stroke EC context marker, not direct receptor":
        return (
            "useful for defining stroke endothelial state, but not prioritised "
            "as direct shuttle receptor"
        )

    if priority == "reject despite surface annotation":
        return (
            "not prioritised for peptide design because primary biology is "
            "intracellular/stress/cytoskeletal/signalling"
        )

    return "not prioritised in this toy analysis"


candidate_pool["manual_priority"] = candidate_pool["gene"].apply(
    manual_priority
)

candidate_pool["manual_interpretation"] = (
    candidate_pool["gene"]
    .map(manual_notes)
    .fillna("")
)

candidate_pool["final_peptide_design_rationale"] = (
    candidate_pool["manual_priority"]
    .apply(final_rationale)
)

In [ ]:
# -------------------------
# 9.4 Compute omics priority score and final manual ranking
# -------------------------

priority_order = {
    "primary peptide-target candidate": 1,
    "secondary candidate / needs validation": 2,
    "stroke EC context marker, not direct receptor": 3,
    "reject despite surface annotation": 4,
    "not prioritised": 5,
}

candidate_pool["priority_order"] = candidate_pool["manual_priority"].map(
    priority_order
)

candidate_pool["omics_priority_score"] = (
    candidate_pool["mean_D02"].rank(pct=True)
    + candidate_pool["effect_D02_vs_Sham"].rank(pct=True)
    + candidate_pool["log2fc_proxy_D02_vs_Sham"].rank(pct=True)
)

final_manual_table = candidate_pool.sort_values(
    ["priority_order", "omics_priority_score"],
    ascending=[True, False],
).copy()

final_columns = [
    "gene",
    "protein_name",
    "mean_D02",
    "mean_Sham",
    "effect_D02_vs_Sham",
    "log2fc_proxy_D02_vs_Sham",
    "pvalue_D02_vs_Sham",
    "omics_priority_score",
    "stroke_response",
    "subcellular_location_text",
    "surface_accessibility",
    "targetability_comment",
    "manual_priority",
    "manual_interpretation",
    "final_peptide_design_rationale",
    "uniprot_accession",
]

display(final_manual_table[final_columns].head(50))

In [ ]:
# -------------------------
# 9.5 Save prioritised candidate table
# -------------------------

final_manual_path = out_dir / "FINAL_manual_prioritised_EC_peptide_target_candidates.csv"

final_manual_table[final_columns].to_csv(
    final_manual_path,
    index=False,
)

print("Saved prioritised EC peptide-target candidates:", final_manual_path)

## 10. Export final receptor-candidate report table

The final report table reformats the prioritised candidates into a compact, human-readable output for downstream peptide-design notebooks. This table includes the gene candidate, UniProt protein annotation, EC pseudobulk effect statistics, surface-accessibility class, biological interpretation, peptide-design rationale, and caveats.

The output remains a gene-level receptor-candidate shortlist. It does not yet validate isoform-specific topology, extracellular-domain structure, internalisation, human translation, or binding suitability.

In [ ]:
# -------------------------
# 10.1 Build final report table
# -------------------------

final_report_table = final_manual_table.copy()

final_report_table = final_report_table.rename(columns={
    "gene": "gene_candidate",
    "protein_name": "uniprot_canonical_protein_annotation",
    "subcellular_location_text": "uniprot_subcellular_location",
    "surface_accessibility": "putative_accessibility_class",
    "manual_priority": "target_prioritisation_class",
    "manual_interpretation": "biological_interpretation",
})

final_report_table["evidence_level"] = (
    "gene-level transcriptomic candidate with UniProt canonical protein annotation; "
    "requires isoform, extracellular-domain, receptor-complex, mouse-human orthology, "
    "and internalisation validation"
)

final_report_columns = [
    "gene_candidate",
    "uniprot_canonical_protein_annotation",
    "mean_D02",
    "mean_Sham",
    "effect_D02_vs_Sham",
    "log2fc_proxy_D02_vs_Sham",
    "pvalue_D02_vs_Sham",
    "omics_priority_score",
    "stroke_response",
    "uniprot_subcellular_location",
    "putative_accessibility_class",
    "target_prioritisation_class",
    "biological_interpretation",
    "final_peptide_design_rationale",
    "evidence_level",
    "uniprot_accession",
]

display(final_report_table[final_report_columns].head(30))

In [ ]:
# -------------------------
# 10.2 Save final report table
# -------------------------

final_report_path = out_dir / "FINAL_gene_level_EC_peptide_target_candidates_with_caveats.csv"

final_report_table[final_report_columns].to_csv(
    final_report_path,
    index=False,
)

print("Saved final gene-level EC peptide-target candidate report:", final_report_path)

Primary candidates identified in this run included `Ly6a`, `Tm4sf1`, `Ly6e`, `Ecscr`, `Cd200`, `Itgb1`, `Esam`, and `Fxyd5`. Secondary and context candidates were retained for interpretation and future expansion.

## Final note

This notebook exports the receptor-candidate shortlist used by the downstream peptide-design notebooks. The prioritised candidates are gene-level hypotheses, not validated shuttle receptors.

The final report table is saved as:

`../data/processed/FINAL_gene_level_EC_peptide_target_candidates_with_caveats.csv`